In [1]:
pip install pythainlp emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 506.6 kB/s  0:00:34m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 417.7 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [emoji]nlp]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import re
import emoji
from pythainlp.util import normalize

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # 1. ลบ URL และแท็กต่างๆ (ลิงก์เว็บ, @username)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)

    # 2. จัดการ Emoji (แปลงเป็นข้อความ เช่น 😂 -> :face_with_tears_of_joy:)
    # การแปลงเป็น Text ช่วยให้ Transformer อ่านความหมายของอารมณ์จากคำได้ง่ายกว่าตัวภาพ
    text = emoji.demojize(text)

    # 3. จัดระเบียบสระและวรรณยุกต์ภาษาไทยด้วย PyThaiNLP 
    # (เช่น แก้ปัญหาพิมพ์ เ เ ติดกันให้เป็น แ หรือจัดลำดับไม้เอกไม้โทให้ถูกต้อง)
    text = normalize(text)

    # 4. ลบอักขระขยะ (เหลือไว้แค่ ไทย, อังกฤษ, ตัวเลข, และโคลอน : ของอีโมจิ)
    # ^ คือการยกเว้น \w (ตัวอักษร/เลข), \s (ช่องว่าง), ก-๙ (ไทย), และ : 
    text = re.sub(r'[^\w\sก-๙:]', ' ', text)

    # 5. ลบช่องว่างและบรรทัดใหม่ที่ซ้ำซ้อนให้เหลือเคาะเดียว
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 1. โหลดข้อมูลดิบที่ดึงมา
df = pd.read_csv("thai_raw_dataset.csv")

# 2. นำฟังก์ชันคลีนข้อมูลไปใช้ (สร้างคอลัมน์ใหม่ชื่อ clean_text)
df['clean_text'] = df['text'].apply(clean_text)

# 3. เรียงคอลัมน์ใหม่ให้ดูง่าย และบันทึก
df = df[['text', 'clean_text', 'score']]
df.to_csv("thai_clean_dataset_X.csv", index=False, encoding='utf-8-sig')

print("ตัวอย่างข้อมูลก่อนและหลังคลีน:")
print(df[['text', 'clean_text']].head())
print("\n✅ คลีนข้อมูลเสร็จสิ้น! บันทึกไฟล์ thai_clean_dataset_X.csv แล้ว")

ตัวอย่างข้อมูลก่อนและหลังคลีน:
            text      clean_text
0          ดีคับ           ดีคับ
1  เยี่ยมยอดจริง   เยี่ยมยอดจริง
2    ได้เคดิตฟรี     ได้เคดิตฟรี
3         ชอบมาก          ชอบมาก
4             😔​  :pensive_face:

✅ คลีนข้อมูลเสร็จสิ้น! บันทึกไฟล์ thai_clean_dataset_X.csv แล้ว
